# SQLarper's ETL Pipeline

In [ ]:
import html
import json
import os
import random
import re
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
from pathlib import Path

import polars as pl
from dotenv import find_dotenv, load_dotenv
from sqlalchemy import create_engine, text

# Which database to load into: "TEST" or "PROD" 
TARGET = "TEST"
TEST_DB_NAME = "__catcruxis"


RAW_DIR = Path("../data/raw")
WILDCATSYNC_FILE = RAW_DIR / "wildcatsync.xml"
HURT_HUB_FILE = RAW_DIR / "hurthub.json"
DAVIDSON_FILE = RAW_DIR / "davidson.json"

# davidson.edu gives local times without a time zone
LOCAL_TZ = "America/New_York"

# When scraping.py downloaded the raw files. Update this if you rerun scraping.py.
DATE_FOUND = datetime(2026, 11, 4, tzinfo=timezone.utc)

load_dotenv(find_dotenv(usecwd=True))  

if TARGET == "TEST":
    db_url = f"{os.environ['DEV_DB_SERVER']}/{TEST_DB_NAME}"
elif TARGET == "PROD":
    db_url = os.environ["PROD_DB_URL"]
else:
    raise ValueError("TARGET must be 'TEST' or 'PROD'")

db_url = db_url.replace("postgres://", "postgresql://", 1)
engine = create_engine(db_url)

## 2. Extract


### 2a WildcatSync

In [ ]:
EV = "{events}"   # the namespace WildcatSync uses for start, end, location, status, host

items = ET.parse(WILDCATSYNC_FILE).getroot().findall(".//item")

ws_raw = pl.DataFrame(
    [
        {
            "title": it.findtext("title"),
            "link": it.findtext("link"),
            "description_html": it.findtext("description"),
            "category": it.findtext("category"),
            "start": it.findtext(EV + "start"),
            "end": it.findtext(EV + "end"),
            "location": it.findtext(EV + "location"),
            "status": it.findtext(EV + "status"),
            "hosts": [h.text for h in it.findall(EV + "host")],
        }
        for it in items
    ],
    schema_overrides={"hosts": pl.List(pl.String)},
)
print(ws_raw.shape)
ws_raw.head(3)

### 2b. Hurt Hub 

In [ ]:
hh_json = json.loads(HURT_HUB_FILE.read_text(encoding="utf-8"))["events"]

hh_raw = pl.DataFrame(
    [
        {
            "title": e["title"],
            "url": e["url"],
            "description_html": e["description"],
            "utc_start_date": e["utc_start_date"],
            "utc_end_date": e["utc_end_date"],
            "venue": e["venue"].get("venue") if isinstance(e["venue"], dict) else None,
            "organizers": [o["organizer"] for o in e["organizer"]],
            "categories": [c["name"] for c in e["categories"]],
        }
        for e in hh_json
    ],
    schema_overrides={"organizers": pl.List(pl.String), "categories": pl.List(pl.String)},
)
print(hh_raw.shape)
hh_raw

### 2c. Davidson College Official Events

In [ ]:
dv_json = json.loads(DAVIDSON_FILE.read_text(encoding="utf-8"))["events"]

dv_raw = pl.DataFrame(
    [
        {k: e.get(k) for k in ("title", "url", "description", "start", "end", "location", "date_time_text")}
        for e in dv_json
    ]
)
print(dv_raw.shape)
dv_raw.head(3)

### 2d. Profiling
Before fixing anything, we look at what we have: missing values, multi-valued cells, formats and repeats. Each finding becomes a cleaning step in section 3 (labelled F1, F2, ...).

In [ ]:
for name, df in (("WildcatSync", ws_raw), ("Hurt Hub", hh_raw), ("Davidson", dv_raw)):
    print(name, df.null_count().to_dicts()[0])

Cancelled events and hosts per event:



In [ ]:
print(ws_raw["status"].value_counts())
print(ws_raw.select(pl.col("hosts").list.len().alias("hosts_per_event"))["hosts_per_event"].value_counts().sort("hosts_per_event"))

Categories:


In [ ]:
print(ws_raw["category"].value_counts().sort("count", descending=True))
print(hh_raw.select(pl.col("categories").explode())["categories"].value_counts())

Date formats:


In [ ]:
print("WildcatSync:", ws_raw["start"][0])
print("Hurt Hub:   ", hh_raw["utc_start_date"][0])
print("Davidson:   ", dv_raw["start"][0])

Location text:



In [ ]:
all_locations = pl.concat([
    ws_raw.select("location"),
    hh_raw.select(pl.col("venue").alias("location")),
    dv_raw.select("location"),
])
print("distinct location strings:", all_locations["location"].n_unique())
all_locations["location"].value_counts().sort("count", descending=True).head(25)

Messy titles:



In [ ]:
print(ws_raw.filter(pl.col("title") != pl.col("title").str.strip_chars()).select("title").head())
print(hh_raw.filter(pl.col("title").str.contains("&")).select("title"))

The same event on two sources, or posted twice:



In [ ]:
def title_key(col):
    return pl.col(col).str.to_lowercase().str.replace_all(r"[^a-z0-9]", "")

ws_keys = ws_raw.select(title_key("title").alias("k"), "start")
dv_keys = dv_raw.select(title_key("title").alias("k"))
print("on both WildcatSync and davidson.edu:")
print(ws_keys.join(dv_keys, on="k", how="inner").select("k").unique())
print("posted twice on WildcatSync (same title and start):")
print(ws_keys.filter(pl.struct("k", "start").is_duplicated()))

## 3. Transform: clean

Each step fixes one finding from the profiling (F1–F11). First, two helpers that all three sources use.

In [ ]:
def strip_html(value):
    # F2/F3: remove tags, decode codes like &amp; and squeeze whitespace
    if value is None:
        return None
    no_tags = re.sub(r"<[^>]+>", " ", value)
    cleaned = " ".join(html.unescape(no_tags).split())
    return cleaned or None


def clean_text(expr):
    # strip spaces at both ends and collapse runs of spaces inside
    return expr.str.strip_chars().str.replace_all(r"\s+", " ")

### 3a. WildcatSync and “F1 drop cancelled; F2/F3 clean title and description (keep only the p-description part of the HTML card); F4 parse GMT dates as UTC; F6 split CamelCase categories.”

In [ ]:
def ws_description(card_html):
    # The HTML card has a <div class="p-description ..."> holding the real text
    match = re.search(r'class="p-description[^"]*">(.*?)</div>', card_html or "", re.S)
    return strip_html(match.group(1)) if match else None


ws = (
    ws_raw
    .filter(pl.col("status") == "confirmed")                                        # F1
    .with_columns(
        clean_text(pl.col("title")).alias("title"),                                 # F2
        pl.col("description_html").map_elements(ws_description, return_dtype=pl.String).alias("description"),  # F3
        pl.col("start").str.to_datetime("%a, %d %b %Y %H:%M:%S GMT", time_zone="UTC").alias("start_time"),     # F4
        pl.col("end").str.to_datetime("%a, %d %b %Y %H:%M:%S GMT", time_zone="UTC").alias("end_time"),
        pl.col("category").str.replace_all(r"([a-z])([A-Z])", "${1} ${2}")          # F6
            .map_elements(lambda c: [c], return_dtype=pl.List(pl.String)).alias("categories"),
        pl.col("hosts").alias("organizers"),
        pl.col("link").alias("source_url"),
        pl.lit("WildcatSync").alias("source_name"),
    )
    .select("title", "description", "start_time", "end_time", "location",
            "organizers", "categories", "source_name", "source_url")
)
print(ws.shape)
ws.head(3)

### 3b. Hurt Hub and “F2/F3 decode HTML in titles and descriptions; F4 the utc_* fields are already UTC, so we just mark them as UTC.”

In [ ]:
hh = (
    hh_raw
    .with_columns(
        pl.col("title").map_elements(strip_html, return_dtype=pl.String).alias("title"),                   # F2
        pl.col("description_html").map_elements(strip_html, return_dtype=pl.String).alias("description"),  # F3
        pl.col("utc_start_date").str.to_datetime("%Y-%m-%d %H:%M:%S", time_zone="UTC").alias("start_time"),  # F4
        pl.col("utc_end_date").str.to_datetime("%Y-%m-%d %H:%M:%S", time_zone="UTC").alias("end_time"),
        pl.col("venue").alias("location"),
        pl.col("url").alias("source_url"),
        pl.lit("Hurt Hub").alias("source_name"),
    )
    .select("title", "description", "start_time", "end_time", "location",
            "organizers", "categories", "source_name", "source_url")
)
hh

### 3c. Davidson College Events and “F3 drop the date/time lines repeated at the start of the description; F4 times are local, so we say they’re America/New_York, then convert to UTC; F7 no organizers or categories, so empty lists for now.”

In [ ]:
def dv_description(desc, date_time_text):
    # F3: drop leading paragraphs that just repeat the date and time
    if desc is None:
        return None
    paragraphs = desc.split("\n\n")
    # e.g. "November 04 5:00PM - 6:30PM" repeats as "November 04" then "5:00PM - 6:30PM"
    while paragraphs and paragraphs[0] in (date_time_text or ""):
        paragraphs.pop(0)
    return strip_html("\n\n".join(paragraphs))


dv = (
    dv_raw
    .with_columns(
        clean_text(pl.col("title")).alias("title"),
        pl.struct("description", "date_time_text").map_elements(
            lambda r: dv_description(r["description"], r["date_time_text"]), return_dtype=pl.String
        ).alias("description"),                                                       # F3
        pl.col("start").str.to_datetime("%Y-%m-%dT%H:%M:%S")                          # F4
            .dt.replace_time_zone(LOCAL_TZ).dt.convert_time_zone("UTC").alias("start_time"),
        pl.col("end").str.to_datetime("%Y-%m-%dT%H:%M:%S")
            .dt.replace_time_zone(LOCAL_TZ).dt.convert_time_zone("UTC").alias("end_time"),
        pl.lit([], dtype=pl.List(pl.String)).alias("organizers"),                    # F7
        pl.lit([], dtype=pl.List(pl.String)).alias("categories"),
        pl.col("url").alias("source_url"),
        pl.lit("Davidson College Events").alias("source_name"),
    )
    .select("title", "description", "start_time", "end_time", "location",
            "organizers", "categories", "source_name", "source_url")
)
dv.head(3)

### 3d. Combine the three sources and “Same columns and types, so we stack them: one row per (event, source).”

In [ ]:
cleaned = pl.concat([ws, hh, dv])
print(cleaned.shape)
cleaned.group_by("source_name").len()

### 3e. Organizers and categories (F7, F8)
- F8: map known spellings of the same organizer to one name.
- F7: if an event has no organizer, use the source's own organizer. If it has no category, use `General`.


In [ ]:
# F8: different spellings of the same organizer -> one name
ORGANIZER_ALIASES = {
    "Hurt Hub@Davidson": "The Hurt Hub",
    "The Hurt Hub@Davidson": "The Hurt Hub",
}

# F7: who runs events that list no organizer
FALLBACK_ORGANIZER = {
    "WildcatSync": "Davidson College",
    "Hurt Hub": "The Hurt Hub",
    "Davidson College Events": "Davidson College",
}
FALLBACK_CATEGORY = "General"

cleaned = cleaned.with_columns(
    pl.col("organizers").list.eval(clean_text(pl.element()).replace(ORGANIZER_ALIASES)),
    pl.col("categories").list.eval(clean_text(pl.element())),
).with_columns(
    pl.when(pl.col("organizers").list.len() == 0)
      .then(pl.col("source_name").replace_strict(FALLBACK_ORGANIZER)
            .map_elements(lambda o: [o], return_dtype=pl.List(pl.String)))
      .otherwise(pl.col("organizers"))
      .alias("organizers"),
    pl.when(pl.col("categories").list.len() == 0)
      .then(pl.lit([FALLBACK_CATEGORY], dtype=pl.List(pl.String)))
      .otherwise(pl.col("categories"))
      .alias("categories"),
)
print("events with no organizer:", cleaned.filter(pl.col("organizers").list.len() == 0).height)
print("events with no category: ", cleaned.filter(pl.col("categories").list.len() == 0).height)

### 3f. Locations: building + room (F9)
1. Collect every distinct location string, so each rule runs once per string.
2. Map each string to (building_name, room_number) with a rule table: "if the text contains this word, the building is X". The room is a room number or a named spot.
3. URLs, Zoom links and empty values go to `Online / TBD`. Anything no rule matches keeps its cleaned text as the building, with no room.

We only merge spellings when the text itself names the building. To fix a location later, add a rule here.

In [ ]:
ONLINE = "Online / TBD"

# (word to look for, building name). First match wins, so specific rules go first.
BUILDING_RULES = [
    ("hurt hub", "The Hurt Hub"),
    ("cannon pool", "Baker Sports Complex"),
    ("baker", "Baker Sports Complex"),
    ("oasis", "Alvarez College Union"),
    ("900 room", "Alvarez College Union"),
    ("morcott", "Alvarez College Union"),
    ("union", "Alvarez College Union"),
    ("alvarez", "Alvarez College Union"),
    ("chambers", "Chambers Building"),
    ("hance", "Chambers Building"),
    ("baxter-davidson", "Chambers Building"),
    ("wall", "Wall Academic Center"),
    ("sloan", "Sloan Music Center"),
    ("tyler-tallman", "Sloan Music Center"),
    ("cunn ", "Cunningham Fine Arts Building"),
    ("barber theatre", "Cunningham Fine Arts Building"),
    ("visual arts center", "Visual Arts Center"),
    ("vac ", "Visual Arts Center"),
    ("spencer", "Spencer-Weinstein Center"),
    ("with these hands", "With These Hands Memorial"),
    ("dcpc", "Davidson College Presbyterian Church"),
    ("watson", "Watson Life Sciences Building"),
]

# Named spots inside a building (used when there's no room number)
ROOM_NAMES = [
    "Hance Auditorium", "Baxter-Davidson Room", "Tyler-Tallman Recital Hall", "Barber Theatre",
    "Semans Lecture Hall", "Smith 900 Room", "900 Room", "Morcott Room", "Oasis", "Atrium",
    "Amphitheater", "Ampitheater", "Cannon Pool", "Dance Studio A", "South Lawn", "Parking Lot",
    "Flagpole", "2nd Floor Lounge", "Hamilton W. McKay Atrium",
]


def split_location(raw):
    # F9: turn one free-text location into (building_name, room_number)
    text_ = " ".join((raw or "").split())
    lower = text_.lower()
    if not text_ or lower.startswith("http") or "zoom" in lower or "virtual" in lower:
        return ONLINE, None

    building = next((b for word, b in BUILDING_RULES if word in lower + " "), None)
    if building is None:
        return text_, None            # unknown place: keep the cleaned text, no room

    number = re.search(r"\b(\d{3,4})\b", text_)
    if number:
        return building, number.group(1)
    for name in ROOM_NAMES:
        if name.lower() in lower:
            return building, "Amphitheater" if name == "Ampitheater" else name   # fix one misspelling
    return building, None             # the building in general


# Run the rules once per distinct location ("" stands for a missing location)
raw_locations = cleaned["location"].fill_null("").unique().sort().to_list()
location_map = pl.DataFrame(
    [{"raw_location": raw, "building_name": b, "room_number": r}
     for raw in raw_locations
     for b, r in [split_location(raw)]],
    schema={"raw_location": pl.String, "building_name": pl.String, "room_number": pl.String},
)
print("distinct raw locations:", location_map.height)
print("distinct buildings:    ", location_map["building_name"].n_unique())
location_map.sort("building_name")

In [ ]:
# Attach each event's building and room using the map above
cleaned = (
    cleaned.with_columns(pl.col("location").fill_null("").alias("raw_location"))
    .join(location_map, on="raw_location", how="left")
    .drop("raw_location")
)
print("events without a building:", cleaned.filter(pl.col("building_name").is_null()).height)

### 3g. Official status
No source has `official_status`, so we set it with a rule:
- davidson.edu and Hurt Hub events are posted by the college: `official`.
- A WildcatSync event is `official` if any host looks like a college office (name contains Office, Center, Program, Department...). Otherwise `student_organized`, since most WildcatSync hosts are student clubs.

In [ ]:
OFFICIAL_WORDS = ["office", "center", "program", "department", "institute", "library", "division",
                  "dean", "galleries", "education abroad", "residence life", "student engagement",
                  "four year advising", "hurt hub", "davidson college"]
official_pattern = "|".join(re.escape(w) for w in OFFICIAL_WORDS)

cleaned = cleaned.with_columns(
    pl.when(pl.col("source_name") != "WildcatSync")
      .then(pl.lit("official"))
      .when(pl.col("organizers").list.eval(
          pl.element().str.to_lowercase().str.contains(official_pattern)).list.any())
      .then(pl.lit("official"))
      .otherwise(pl.lit("student_organized"))
      .alias("official_status")
)
cleaned.group_by("source_name", "official_status").len().sort("source_name")

### 3h. Duplicates: one event, many sources (F10)
An event is identified by its title (lowercase, letters and digits only) plus its start time. Rows that share this key are the same event. In Normalize we keep one row of event details (from the highest-priority source: WildcatSync, then Hurt Hub, then Davidson) and one `event_source` row for every source.

In [ ]:
SOURCE_PRIORITY = {"WildcatSync": 1, "Hurt Hub": 2, "Davidson College Events": 3}

cleaned = cleaned.with_columns(
    (pl.col("title").str.to_lowercase().str.replace_all(r"[^a-z0-9]", "")
     + "|" + pl.col("start_time").dt.strftime("%Y-%m-%dT%H:%M")).alias("event_key"),
    pl.col("source_name").replace_strict(SOURCE_PRIORITY, return_dtype=pl.Int64).alias("priority"),
)
dupes = cleaned.filter(pl.col("event_key").is_duplicated()).select("title", "source_name", "start_time")
print("rows that share an event with another row:")
dupes.sort("title")

In [ ]:
# Sanity checks on the clean data. Each one fails loudly if something is off.
assert cleaned.height == ws_raw.filter(pl.col("status") == "confirmed").height + hh_raw.height + dv_raw.height, "rows lost or added"
assert cleaned["title"].null_count() == 0, "an event has no title"
assert cleaned["start_time"].null_count() == 0, "an event has no start time"
assert cleaned.filter(pl.col("end_time") < pl.col("start_time")).height == 0, "an event ends before it starts"
assert not cleaned["title"].str.contains("&#|&amp;").any(), "HTML codes left in a title"
assert not cleaned["description"].str.contains("<[a-z/]").fill_null(False).any(), "HTML tags left in a description"
assert cleaned["building_name"].null_count() == 0, "an event has no building"
assert (cleaned["organizers"].list.len() > 0).all(), "an event has no organizer"
assert (cleaned["categories"].list.len() > 0).all(), "an event has no category"
print("all clean-data checks passed")

## 4. Transform: normalize

We split the clean data into one DataFrame per table, with the same column names as the database (without `created_at`/`modified_at`, which the database fills in).
We give every row its primary key here, so foreign keys can be checked in Polars before anything touches the database. `source` is different: its rows come from the reference rows migration, so we read its IDs from the database.

In [ ]:
with engine.connect() as conn:
    source = pl.read_database("SELECT source_id, source_name, source_type FROM source ORDER BY source_id", conn)
all_rows = cleaned   # every (event, source) row
source

4a. building and location

In [ ]:
building = (
    all_rows.select("building_name").unique().sort("building_name")
    .with_row_index("building_id", offset=1)
    .with_columns(
        pl.col("building_id").cast(pl.Int32),
        # No source has coordinates, so they stay empty for now
        pl.lit(None, dtype=pl.Decimal(9, 6)).alias("latitude"),
        pl.lit(None, dtype=pl.Decimal(9, 6)).alias("longitude"),
    )
)

# A text key for (building, room). Joining on it avoids comparing NULL rooms.
def location_key():
    return pl.col("building_name") + "|" + pl.col("room_number").fill_null("")

all_rows = all_rows.with_columns(location_key().alias("location_key"))

location = (
    all_rows.select("building_name", "room_number", "location_key").unique()
    .join(building.select("building_id", "building_name"), on="building_name")
    .sort("building_name", "room_number", nulls_last=True)
    .with_row_index("location_id", offset=1)
    .with_columns(pl.col("location_id").cast(pl.Int32))
)
print(building.height, "buildings,", location.height, "locations")

4b. “One row per event_key. When an event came from several sources, its details come from the highest-priority source.”

In [ ]:
event_rows = (
    all_rows.sort("priority")
    .unique(subset="event_key", keep="first", maintain_order=True)
    .sort("start_time", "title")
    .with_row_index("event_id", offset=1)
    .with_columns(pl.col("event_id").cast(pl.Int32))
    .join(location.select("location_id", "location_key"), on="location_key", how="left")
)
event = event_rows.select("event_id", "location_id", "title", "description",
                          "start_time", "end_time", "official_status")

# Every (event, source) row with its event_id. The bridge tables use this.
rows_with_ids = all_rows.join(event_rows.select("event_key", "event_id"), on="event_key")
print(event.height, "events from", all_rows.height, "source rows")
event.head()

4c. Lists are exploded: an event with 3 hosts becomes 3 event_organizer rows. For an event found on two sources, if one gave a real organizer and the other only the fallback (F7), we drop the fallback.

In [ ]:
def drop_fallback(links, name_col, fallback_names):
    # Keep a fallback value only if it's the event's only value
    is_fallback = pl.col(name_col).is_in(fallback_names)
    has_real = (~is_fallback).any().over("event_id")
    return links.filter(~(is_fallback & has_real))


event_organizer_links = drop_fallback( 
    rows_with_ids.select("event_id", pl.col("organizers").alias("organizer_name")).explode("organizer_name").unique(),
    "organizer_name", list(set(FALLBACK_ORGANIZER.values())),
)
event_category_links = drop_fallback(
    rows_with_ids.select("event_id", pl.col("categories").alias("category_name")).explode("category_name").unique(),
    "category_name", [FALLBACK_CATEGORY],
)

organizer = (
    event_organizer_links.select("organizer_name").unique().sort("organizer_name")
    .with_row_index("organizer_id", offset=1)
    .with_columns(
        pl.col("organizer_id").cast(pl.Int32),
        pl.lit(None, dtype=pl.String).alias("organizer_type"),   # no source has it
    )
)

category = (
    event_category_links.select("category_name").unique().sort("category_name")
    .with_row_index("category_id", offset=1)
    .with_columns(pl.col("category_id").cast(pl.Int32))
)

event_organizer = (
    event_organizer_links
    .join(organizer.select("organizer_id", "organizer_name"), on="organizer_name")
    .select("event_id", "organizer_id").unique().sort("event_id", "organizer_id")
    .with_row_index("event_organizer_id", offset=1)
    .with_columns(pl.col("event_organizer_id").cast(pl.Int32))
)

event_category = (
    event_category_links
    .join(category.select("category_id", "category_name"), on="category_name")
    .select("event_id", "category_id").unique().sort("event_id", "category_id")
    .with_row_index("event_category_id", offset=1)
    .with_columns(pl.col("event_category_id").cast(pl.Int32))
)


event_source = (
    rows_with_ids.join(source.select("source_id", "source_name"), on="source_name")
    .select("event_id", "source_id", "source_url")
    .unique(subset=["event_id", "source_id"], keep="first", maintain_order=True)
    .sort("event_id", "source_id")
    .with_row_index("event_source_id", offset=1)  
    .with_columns(
        pl.col("event_source_id").cast(pl.Int32),
        pl.lit(DATE_FOUND).alias("date_found"),   # when we downloaded it
    )
)

# Keep only the database's columns, in its order
building = building.select("building_id", "building_name", "latitude", "longitude")
location = location.select("location_id", "building_id", "room_number")

tables = {
    "building": building, "location": location, "organizer": organizer, "category": category,
    "event": event, "event_organizer": event_organizer, "event_category": event_category,
    "event_source": event_source,
}
for name, df in tables.items():
    print(f"{name:16} {df.height:4} rows   {df.columns}")

### 4d. Checks before loading
The database would reject bad rows anyway, but a failed check here is easier to debug than an error halfway through a load.
1. Primary keys: no duplicates, no nulls.
2. Unique rules: the same ones the database has.
3. Referential integrity: every foreign key exists in the table it points to.
4. At least one: every event has an organizer, a category and a source.
5. Nothing lost: joining the tables back together gives back every (event, source) pair.

In [ ]:
def check(condition, message):
    assert condition, "FAILED: " + message
    print("ok  ", message)


# 1. Primary keys
for name, df in tables.items():
    pk = f"{name}_id"
    check(df[pk].null_count() == 0 and df[pk].is_unique().all(), f"{name}.{pk} unique and not null")

# 2. Unique rules from the schema
check(building["building_name"].is_unique().all(), "building_name unique")
check(not location.select("building_id", "room_number").is_duplicated().any(), "(building_id, room_number) unique")
check(organizer["organizer_name"].is_unique().all(), "organizer_name unique")
check(category["category_name"].is_unique().all(), "category_name unique")
for name, cols in (("event_organizer", ["event_id", "organizer_id"]),
                   ("event_category", ["event_id", "category_id"]),
                   ("event_source", ["event_id", "source_id"])):
    check(not tables[name].select(cols).is_duplicated().any(), f"{name} pair unique")

# 3. Referential integrity: (child table, column, parent table, column)
fks = [
    (location, "building_id", building, "building_id"),
    (event, "location_id", location, "location_id"),
    (event_organizer, "event_id", event, "event_id"),
    (event_organizer, "organizer_id", organizer, "organizer_id"),
    (event_category, "event_id", event, "event_id"),
    (event_category, "category_id", category, "category_id"),
    (event_source, "event_id", event, "event_id"),
    (event_source, "source_id", source, "source_id"),
]
for child, col, parent, pcol in fks:
    check(child[col].null_count() == 0 and child[col].is_in(parent[pcol].implode()).all(), f"{col} exists in parent")

# 4. Every event has at least one organizer, category and source
for bridge_name in ("event_organizer", "event_category", "event_source"):
    check(event["event_id"].is_in(tables[bridge_name]["event_id"].implode()).all(), f"every event has a row in {bridge_name}")

# The CHECK constraints in the schema
check(event.filter(pl.col("end_time") < pl.col("start_time")).height == 0, "end_time >= start_time")
check(event["official_status"].is_in(["official", "student_organized"]).all(), "official_status allowed")

In [ ]:
# 5. Rebuild every (event, source) pair from the tables and compare
rebuilt = (
    event_source.join(event, on="event_id").join(source, on="source_id")
    .join(location, on="location_id").join(building, on="building_id")
    .select("event_id", "source_name")
)
expected = rows_with_ids.select("event_id", "source_name").unique()
check(rebuilt.height == expected.height, f"every (event, source) pair survived ({rebuilt.height})")
check(rebuilt.join(expected, on=["event_id", "source_name"], how="anti").height == 0, "no extra (event, source) pairs")
check(event_organizer_links.height == event_organizer.height, f"every organizer link survived ({event_organizer.height})")
check(event_category_links.height == event_category.height, f"every category link survived ({event_category.height})")

## 5. Load
- **Parents first:** building → location → event → bridge tables.
- **One transaction:** everything is inside `engine.begin()`. If any insert fails, the whole load is undone.
- **Append only:** the migrations own the tables; we only insert rows.
- **Safe to rerun:** we empty the tables this notebook fills (`TRUNCATE ... RESTART IDENTITY`) and reload them. We never truncate `source`, because its rows come from a migration.

Because we truncate and reload, `created_at` means the time of the latest load.

Our IDs are `GENERATED ALWAYS AS IDENTITY`, which normally refuses IDs you supply. We insert the IDs we checked in section 4, so we use `OVERRIDING SYSTEM VALUE`. Afterwards we move each ID counter past the highest ID, so future inserts don't collide.

In [ ]:
LOAD_ORDER = ["building", "location", "organizer", "category", "event",
              "event_organizer", "event_category", "event_source"]   # never includes source


def insert(conn, table, df):
    cols = ", ".join(df.columns)
    params = ", ".join(f":{c}" for c in df.columns)
    sql = text(f"INSERT INTO {table} ({cols}) OVERRIDING SYSTEM VALUE VALUES ({params})")
    conn.execute(sql, df.to_dicts())


with engine.begin() as conn:   # one transaction: all or nothing
    conn.execute(text(f"TRUNCATE {', '.join(LOAD_ORDER)} RESTART IDENTITY"))
    for table in LOAD_ORDER:
        insert(conn, table, tables[table])
        print(f"loaded {table:16} {tables[table].height:4} rows")
    for table in LOAD_ORDER:
        # move the ID counter past the highest ID we inserted
        conn.execute(text(
            f"SELECT setval(pg_get_serial_sequence('{table}', '{table}_id'), "
            f"COALESCE((SELECT MAX({table}_id) FROM {table}), 1))"
        ))
print("Load committed.")

### 5a. Verify
Each table's row count in the database must equal its DataFrame. Then one analyst-style query joining several tables.

In [ ]:
with engine.connect() as conn:
    counts = []
    for table in LOAD_ORDER + ["source"]:
        in_db = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar_one()
        expected_rows = source.height if table == "source" else tables[table].height
        counts.append({"table": table, "dataframe": expected_rows, "database": in_db, "match": in_db == expected_rows})
counts = pl.DataFrame(counts)
assert counts["match"].all(), "row counts don't match"
counts

In [ ]:
query = """
SELECT c.category_name,
       s.source_name,
       COUNT(DISTINCT e.event_id)    AS events,
       COUNT(DISTINCT b.building_id) AS buildings
FROM event AS e
JOIN event_category AS ec ON ec.event_id = e.event_id
JOIN category       AS c  ON c.category_id = ec.category_id
JOIN event_source   AS es ON es.event_id = e.event_id
JOIN source         AS s  ON s.source_id = es.source_id
JOIN location       AS l  ON l.location_id = e.location_id
JOIN building       AS b  ON b.building_id = l.building_id
GROUP BY c.category_name, s.source_name
ORDER BY events DESC
LIMIT 15
"""
with engine.connect() as conn:
    result = pl.read_database(query, conn)
result